# realSEUDO streaming walkthrough

This notebook walks through running realSEUDO's **streaming / online** ROI-discovery
pipeline (`seudo.streaming.realSEUDOfit`) frame by frame on a real two-photon
calcium-imaging movie, starting from **no prior knowledge of cell locations**.

It covers:

1. **Loading the movie** (`code/demoData1.mat`, via a lazy HDF5 loader so the whole
   movie is never held in memory at once).
2. **Configuring the streaming pipeline** with the same parameters used in production
   (`scripts/run_realseudo_full_movie.py`).
3. **Running the discovery loop**, with a live-updating figure showing:
   - the current frame, with **confirmed cells (Xstab)** outlined in solid color and
     **in-progress candidate ROIs (Xtemp)** outlined in dashed orange,
   - the **time traces** of every confirmed cell's activity discovered so far.

By default this demo processes only the first `N_DEMO_FRAMES` frames of the movie
(fast enough to run interactively) -- see the note at the end for running the full
41,756-frame movie.

In [ ]:
import sys
sys.path.insert(0, '..')  # so `import seudo` finds python/seudo from notebooks/

import colorsys

import h5py
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display

from seudo.matlab_io import Hdf5Movie, load_profiles
from seudo.streaming import (
    DetectionParams, FitParams, PromotionParams, StreamingState,
    realSEUDOfit, _build_promoted_profile,
)

%matplotlib inline

## 1. Load the data

`code/demoData1.mat` bundles the raw movie (`M`) alongside CNMF-derived ground truth
(`P` profiles) that we only load here for reference -- `realSEUDOfit` never sees them.

The movie is wrapped in `Hdf5Movie`, a thin lazy loader that reads one frame at a time
straight from disk (the full movie is 41,756 frames and doesn't comfortably fit in
memory all at once).

In [ ]:
DEMO_PATH = '../../code/demoData1.mat'

demo_file = h5py.File(DEMO_PATH, 'r')
movie = Hdf5Movie(demo_file['M'])
cnmf_profiles = load_profiles(demo_file)  # reference only, not used by realSEUDOfit

mov_y, mov_x, n_frames_total = movie.shape
print(f'movie: {mov_y}x{mov_x} pixels, {n_frames_total} frames')
print(f'CNMF ground truth: {cnmf_profiles.shape[2]} cells (for reference only, not used below)')

## 2. Configure the streaming pipeline

`realSEUDOfit(frame, state)` is called once per incoming frame and mutates `state` in
place: it fits every already-known cell's activity against the frame, looks for new
candidate ROIs (**Xtemp**) in what's left unexplained, tracks those candidates across
frames, and promotes a candidate to a confirmed cell (**Xstab**) once it's been
consistently detected for `consecutive_frames_required` frames.

The parameters below are the current production defaults from
`scripts/run_realseudo_full_movie.py` -- see that script's comments for the full
tuning history behind each one (temporal averaging window, candidate-exclusion
radius, anti-duplication merge thresholds, and a small shape-smoothing pass applied
to candidate profiles).

In [ ]:
# same production defaults as scripts/run_realseudo_full_movie.py
SEUDO_PARAMS = dict(sigma2=0.0020, lambda_blob=10.0, blob_radius=3.0, pad_space=5,
                     n_jobs=8, native_nthreads=4, blob_spacing=3.0,
                     spatial_denoise_radius=None, lookahead_frames=5)
DETECTION_PARAMS = dict(exclude_radius_known_cells=-1, noise_grid_shape=(2, 2),
                         candidate_profile_threshold=0.05, xtemp_smooth_sigma=0.5)
PROMOTION_PARAMS = dict(consecutive_frames_required=3, eq8_merge_threshold=0.2, eq9_merge_threshold=0.2)

state = StreamingState(
    (mov_y, mov_x),
    fit=FitParams(**SEUDO_PARAMS),
    detection=DetectionParams(**DETECTION_PARAMS),
    promotion=PromotionParams(**PROMOTION_PARAMS),
)

## 3. Run the discovery loop

`lookahead_frames=5` above means `realSEUDOfit` buffers incoming frames and only
reports a result once 4 frames beyond the current one have arrived -- it returns
`None` for the first few calls, and every reported `result.frame_index` lags the
raw input frame by a few frames. This trades a small amount of latency for
measurably better detection quality (see `FitParams.lookahead_frames`'s docstring).

Every `UPDATE_EVERY` frames, the figure below redraws with two panels:

- **left**: the current raw frame, with every confirmed cell's footprint outlined
  in its own color, and every in-progress candidate ROI outlined in dashed orange.
- **right**: the discovered activity trace of every confirmed cell so far, stacked
  vertically (same colors as the left panel) so newer/weaker cells stay legible
  next to earlier/stronger ones.

In [ ]:
# n perceptually well-separated colors via golden-angle hue stepping --
# deterministic per index, so a given cell_id keeps the same color across
# redraws even as n (the total cell count) grows
def distinct_colors(n):
    golden = 0.6180339887498949
    hue = 0.0
    colors = []
    for _ in range(n):
        colors.append(colorsys.hsv_to_rgb(hue, 0.85, 0.95))
        hue = (hue + golden) % 1.0
    return colors


def draw_profile_contour(ax, profile, color, level=0.2, linestyle='-'):
    peak = profile.max()
    if peak <= 0:
        return
    ax.contour(profile / peak, levels=[level], colors=[color], linewidths=1.5, linestyles=linestyle)

In [ ]:
N_DEMO_FRAMES = 1200   # bump up to movie.shape[2] for the full movie -- see the note below
UPDATE_EVERY = 100      # redraw the figure every this many input frames

activity_by_frame = {}   # result.frame_index -> {cell_id: activity}

fig, (ax_spatial, ax_traces) = plt.subplots(1, 2, figsize=(13, 6))

for ff in range(N_DEMO_FRAMES):
    frame = movie.get_frame(ff)
    result = realSEUDOfit(frame, state)
    if result is not None:
        activity_by_frame[result.frame_index] = result.activity

    if (ff + 1) % UPDATE_EVERY != 0 and ff != N_DEMO_FRAMES - 1:
        continue

    n_cells = state.profiles.shape[2]
    colors = distinct_colors(max(n_cells, 1))

    ax_spatial.clear()
    ax_traces.clear()

    # -- left: current frame, confirmed cells (solid) + candidate ROIs (dashed orange) --
    ax_spatial.imshow(frame, cmap='gray',
                       vmin=np.percentile(frame, 1), vmax=np.percentile(frame, 99.5))
    for cell_id in range(n_cells):
        draw_profile_contour(ax_spatial, state.profiles[:, :, cell_id], colors[cell_id])
    for track in state.candidate_tracks.values():
        xtemp_profile, _bbox = _build_promoted_profile(
            track, (state.mov_y, state.mov_x), smooth_sigma=state.detection.xtemp_smooth_sigma)
        draw_profile_contour(ax_spatial, xtemp_profile, color='orange', linestyle='--')
    ax_spatial.set_title(f'frame {ff + 1}/{N_DEMO_FRAMES}  --  {n_cells} confirmed cell(s), '
                          f'{len(state.candidate_tracks)} candidate ROI(s) (dashed orange)')
    ax_spatial.axis('off')

    # -- right: every confirmed cell's activity trace so far, stacked with an offset --
    frame_indices = sorted(activity_by_frame.keys())
    offset = 0.0
    for cell_id in range(n_cells):
        trace = np.array([activity_by_frame[fidx].get(cell_id, np.nan) for fidx in frame_indices])
        scale = np.nanmax(np.abs(trace)) if np.any(~np.isnan(trace)) else 1.0
        ax_traces.plot(frame_indices, trace + offset, color=colors[cell_id], linewidth=1.0)
        offset += 1.2 * (scale if scale > 0 else 1.0)
    ax_traces.set_title("discovered cells' time traces so far (stacked)")
    ax_traces.set_xlabel('frame')
    ax_traces.set_yticks([])

    fig.tight_layout()
    clear_output(wait=True)
    display(fig)

state.close()  # release the per-frame fitting thread pool
print(f'processed {N_DEMO_FRAMES} frames, {state.profiles.shape[2]} cell(s) discovered')

## Running the full movie

Set `N_DEMO_FRAMES = movie.shape[2]` (all 41,756 frames) to reproduce the actual
production run -- expect roughly 45-70 minutes at these settings on this dataset.
You'll likely also want to raise `UPDATE_EVERY` so redrawing the figure doesn't
dominate wall-clock time.

`scripts/run_realseudo_full_movie.py` is the actual script used to produce the
production comparison results under `scripts/output/` (contour overlays, per-ROI
grids, and time-course comparisons against the CNMF ground truth) -- use it directly
for a full non-interactive run, and come back to this notebook for interactive
exploration of a shorter window.

In [ ]:
demo_file.close()